# DataSense Final Test Notebook — LLM + GPU backend benchmark

This notebook tests the exact question you are trying to answer:

- give the LLM a dataset
- ask it to solve a real task
- compare **pandas/sklearn** vs **cuDF/cuML**
- optionally test **`cudf.pandas`** as a no-retraining workaround

The outcome should tell you which problem statement is worth shipping in 2 days.


## What this notebook decides

It compares end-to-end latency for tasks that matter in the NVIDIA track:

- ranking / triage
- rolling-window alerting
- aggregation / dashboard backends
- prediction / scoring

The key metric is not only model speed. It is the **full task latency**:
LLM generation + code execution + result production.


In [ ]:
%%capture
!pip -q install --upgrade pandas numpy matplotlib scikit-learn nbformat
!pip -q install --extra-index-url=https://pypi.nvidia.com cudf-cu12 cuml-cu12 cupy-cuda12x
!pip -q install transformers accelerate sentencepiece


In [ ]:
import os, re, time, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import cudf
    import cuml
    RAPIDS_OK = True
except Exception as e:
    RAPIDS_OK = False
    cudf = None
    cuml = None
    print("RAPIDS import failed:", e)

print("RAPIDS_OK =", RAPIDS_OK)

MODEL_ID_OR_PATH = os.environ.get("DATASENSE_MODEL", "").strip()
DATA_PATH = os.environ.get("DATASENSE_DATA", "").strip()
N_ROWS = int(os.environ.get("DATASENSE_ROWS", "1000000"))
MAX_NEW_TOKENS = int(os.environ.get("DATASENSE_MAX_NEW_TOKENS", "512"))
TEMPERATURE = float(os.environ.get("DATASENSE_TEMPERATURE", "0.0"))

if not MODEL_ID_OR_PATH:
    print("Set DATASENSE_MODEL to your fine-tuned DataSense model path or Hugging Face repo.")


In [ ]:
def timer(fn, *args, **kwargs):
    t0 = time.perf_counter()
    out = fn(*args, **kwargs)
    return out, time.perf_counter() - t0

def extract_code(text):
    text = text.strip()
    m = re.search(r"```(?:python)?\s*(.*?)```", text, flags=re.S | re.I)
    if m:
        return m.group(1).strip()
    return text

def compact_schema(df, max_cols=18):
    lines = [f"- rows: {len(df):,}", f"- cols: {len(df.columns)}", "- schema:"]
    for c in list(df.columns[:max_cols]):
        lines.append(f"  - {c}: {df[c].dtype}")
    if len(df.columns) > max_cols:
        lines.append(f"  - ... {len(df.columns) - max_cols} more columns")
    return "\n".join(lines)


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tokenizer = None
model = None

def load_model(model_id_or_path):
    tok = AutoTokenizer.from_pretrained(model_id_or_path, use_fast=True)
    mdl = AutoModelForCausalLM.from_pretrained(
        model_id_or_path,
        device_map="auto",
        torch_dtype="auto",
        trust_remote_code=True,
    )
    return tok, mdl

def llm_generate(prompt, system_prompt, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE):
    if tokenizer is None or model is None:
        raise RuntimeError("Model not loaded.")
    if hasattr(tokenizer, "apply_chat_template"):
        msgs = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ]
        full_prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    else:
        full_prompt = system_prompt + "\n\n" + prompt
    inputs = tokenizer(full_prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0),
            temperature=temperature if temperature > 0 else None,
            pad_token_id=getattr(tokenizer, "eos_token_id", None),
        )
    txt = tokenizer.decode(out[0], skip_special_tokens=True)
    if txt.startswith(full_prompt):
        txt = txt[len(full_prompt):]
    return txt.strip()


In [ ]:
if not MODEL_ID_OR_PATH:
    raise RuntimeError("Set DATASENSE_MODEL before running.")
tokenizer, model = load_model(MODEL_ID_OR_PATH)
print("Model loaded:", MODEL_ID_OR_PATH)


In [ ]:
def make_synthetic_dataset(n_rows=N_ROWS, seed=42):
    rng = np.random.default_rng(seed)
    dates = pd.Timestamp("2024-01-01") + pd.to_timedelta(rng.integers(0, 365 * 24 * 3600, n_rows), unit="s")
    df = pd.DataFrame({
        "txn_id": np.arange(n_rows, dtype=np.int64),
        "date": pd.to_datetime(dates).floor("D"),
        "store_id": rng.integers(0, 250, n_rows, dtype=np.int32),
        "product_id": rng.integers(0, 2000, n_rows, dtype=np.int32),
        "user_id": rng.integers(0, 50000, n_rows, dtype=np.int32),
        "region": rng.integers(0, 8, n_rows, dtype=np.int8),
        "qty": rng.integers(1, 10, n_rows, dtype=np.int16),
        "price": rng.gamma(2.0, 20.0, n_rows).round(2),
        "discount_pct": rng.uniform(0, 0.35, n_rows).round(3),
        "return_flag": (rng.random(n_rows) < 0.05).astype(np.int8),
        "support_tier": rng.integers(1, 5, n_rows, dtype=np.int8),
        "inventory": rng.integers(0, 5000, n_rows, dtype=np.int32),
        "days_since_restock": rng.integers(0, 60, n_rows, dtype=np.int16),
        "session_minutes": rng.gamma(3.0, 5.0, n_rows).round(2),
        "ticket_age_hours": rng.gamma(2.0, 8.0, n_rows).round(2),
        "sentiment": rng.normal(0, 1, n_rows).round(3),
    })
    df["revenue"] = (df["qty"] * df["price"] * (1 - df["discount_pct"])).round(2)
    df["margin"] = (df["revenue"] * rng.uniform(0.12, 0.42, n_rows)).round(2)
    df["risk_score"] = (
        0.45 * df["return_flag"].astype(float)
        + 0.20 * (df["ticket_age_hours"] / (df["ticket_age_hours"].max() + 1e-9))
        + 0.20 * (1 - np.clip(df["sentiment"], -2, 2) / 2)
        + 0.15 * (df["days_since_restock"] / (df["days_since_restock"].max() + 1e-9))
    )
    df["risk_label"] = (df["risk_score"] > df["risk_score"].quantile(0.75)).astype(np.int8)
    return df

if DATA_PATH and os.path.exists(DATA_PATH):
    pdf = pd.read_csv(DATA_PATH)
    print("Loaded CSV:", DATA_PATH)
else:
    pdf = make_synthetic_dataset()
    print("Using synthetic dataset.")

if RAPIDS_OK:
    gdf = cudf.DataFrame.from_pandas(pdf)
else:
    gdf = None

pdf.shape


In [ ]:
from dataclasses import dataclass
from typing import List

@dataclass
class TaskSpec:
    name: str
    user_question: str
    notes: str

TASKS: List[TaskSpec] = [
    TaskSpec(
        name="priority_ranking",
        user_question="Rank the top 25 rows by operational priority using return_flag, ticket_age_hours, days_since_restock, sentiment, and margin. Return a dataframe with a priority_score column.",
        notes="Triage / ranking / decision support"
    ),
    TaskSpec(
        name="rolling_alert",
        user_question="Compute a 7-day rolling sales trend by store and flag the top 10 biggest negative anomalies where current revenue is below the rolling average.",
        notes="Freshness / alerting / time-series"
    ),
    TaskSpec(
        name="dashboard_summary",
        user_question="Create an aggregated dashboard summary by region and store tier. Show revenue, margin, return rate, and average ticket age in a compact table.",
        notes="BI / dashboard backend"
    ),
    TaskSpec(
        name="risk_model",
        user_question="Train a fast model to predict risk_label from the numeric columns, then return the top 20 highest-risk rows with predicted risk probability.",
        notes="Scoring / classification"
    ),
]

SYSTEM_PROMPT = (
    "You are a data analysis agent. "
    "Write only executable Python code. Do not include markdown or explanation. "
    "Use the provided dataframe variable named df. "
    "For CPU mode, prefer pandas and sklearn. "
    "For GPU mode, prefer cuDF and cuML. "
    "End by assigning the final answer to a variable named result."
)

def build_prompt(task, schema_text, backend):
    backend_hint = {
        "pandas": "Use pandas + sklearn only.",
        "cudf": "Use cuDF + cuML only. Prefer GPU-native dataframe operations.",
    }[backend]
    return textwrap.dedent(f"""
    Task:
    {task.user_question}

    Dataset schema:
    {schema_text}

    Constraints:
    - {backend_hint}
    - Use df as the input dataframe.
    - Return a result variable that is a dataframe or model output that supports the task.
    """).strip()


In [ ]:
def run_generated_code(code, df_obj):
    code = extract_code(code)
    ns = {
        "__builtins__": __builtins__,
        "pd": pd,
        "np": np,
        "df": df_obj,
        "result": None,
    }
    if RAPIDS_OK:
        ns["cudf"] = cudf
        ns["cuml"] = cuml
    try:
        from sklearn.model_selection import train_test_split
        from sklearn.linear_model import LinearRegression, LogisticRegression
        from sklearn.ensemble import RandomForestClassifier
        from sklearn.metrics import roc_auc_score, accuracy_score
        ns.update({
            "train_test_split": train_test_split,
            "LinearRegression": LinearRegression,
            "LogisticRegression": LogisticRegression,
            "RandomForestClassifier": RandomForestClassifier,
            "roc_auc_score": roc_auc_score,
            "accuracy_score": accuracy_score,
        })
    except Exception:
        pass
    exec(code, ns, ns)
    return ns.get("result"), code

def normalize_obj(obj):
    if obj is None:
        return None
    if RAPIDS_OK and isinstance(obj, cudf.DataFrame):
        return obj.to_pandas()
    if hasattr(obj, "to_pandas") and not isinstance(obj, pd.DataFrame):
        try:
            return obj.to_pandas()
        except Exception:
            return obj
    return obj

def compare_outputs(a, b):
    a = normalize_obj(a)
    b = normalize_obj(b)
    out = {"same_type": type(a).__name__ == type(b).__name__}
    if isinstance(a, pd.DataFrame) and isinstance(b, pd.DataFrame):
        out["same_shape"] = a.shape == b.shape
        common = [c for c in a.columns if c in b.columns]
        out["common_cols"] = len(common)
        if common:
            out["head_equal"] = a[common].head(20).reset_index(drop=True).equals(
                b[common].head(20).reset_index(drop=True)
            )
        else:
            out["head_equal"] = False
    else:
        out["same_shape"] = None
        out["common_cols"] = None
        out["head_equal"] = None
    return out


In [ ]:
BACKENDS = ["pandas"] + (["cudf"] if RAPIDS_OK else [])
schema_text = compact_schema(pdf)
benchmark_rows = []

for task in TASKS:
    print(f"\n=== {task.name} ===")
    print(task.notes)
    for backend in BACKENDS:
        df_obj = pdf.copy() if backend == "pandas" else cudf.DataFrame.from_pandas(pdf)
        prompt = build_prompt(task, schema_text, backend)

        t0 = time.perf_counter()
        raw = llm_generate(prompt, SYSTEM_PROMPT)
        gen_s = time.perf_counter() - t0

        code = extract_code(raw)
        print(f"\n--- {backend.upper()} generated code ---")
        print(code[:1200])

        err = None
        out = None
        exec_s = None
        try:
            out, exec_s = timer(run_generated_code, code, df_obj)
        except Exception as e:
            err = f"{type(e).__name__}: {e}"
            print("Execution failed:", err)

        benchmark_rows.append({
            "task": task.name,
            "backend": backend,
            "generation_s": gen_s,
            "execution_s": exec_s,
            "total_s": gen_s + (exec_s or 0.0),
            "error": err,
            "code": code,
            "result": normalize_obj(out) if out is not None else None,
        })

        print(f"{backend}: gen={gen_s:.2f}s exec={exec_s if exec_s is not None else 'ERR'} total={gen_s + (exec_s or 0.0):.2f}s")


In [ ]:
bench_df = pd.DataFrame([{k: v for k, v in row.items() if k not in ("code", "result")} for row in benchmark_rows])
display(bench_df)

speedup_rows = []
for task in TASKS:
    sub = bench_df[(bench_df["task"] == task.name) & bench_df["error"].isna()]
    if set(["pandas", "cudf"]).issubset(set(sub["backend"])):
        p = sub[sub["backend"] == "pandas"].iloc[0]
        g = sub[sub["backend"] == "cudf"].iloc[0]
        speedup_rows.append({
            "task": task.name,
            "gen_speedup_x": p["generation_s"] / g["generation_s"] if g["generation_s"] else None,
            "exec_speedup_x": p["execution_s"] / g["execution_s"] if g["execution_s"] else None,
            "total_speedup_x": p["total_s"] / g["total_s"] if g["total_s"] else None,
        })

speedup_df = pd.DataFrame(speedup_rows)
display(speedup_df)

comparison_rows = []
for task in TASKS:
    p = next((r for r in benchmark_rows if r["task"] == task.name and r["backend"] == "pandas" and r["error"] is None), None)
    g = next((r for r in benchmark_rows if r["task"] == task.name and r["backend"] == "cudf" and r["error"] is None), None)
    if p and g:
        comparison_rows.append({"task": task.name, **compare_outputs(p["result"], g["result"])})
display(pd.DataFrame(comparison_rows))


In [ ]:
if not bench_df.empty:
    fig, ax = plt.subplots(figsize=(11, 5))
    for backend in bench_df["backend"].unique():
        sub = bench_df[bench_df["backend"] == backend]
        ax.plot(sub["task"], sub["total_s"], marker="o", label=backend)
    ax.set_ylabel("seconds")
    ax.set_title("End-to-end LLM task latency")
    ax.legend()
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()

if not speedup_df.empty:
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.bar(speedup_df["task"], speedup_df["total_speedup_x"])
    ax.set_ylabel("CPU total time / GPU total time")
    ax.set_title("Total latency speedup by task")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()


In [ ]:
def decide_problem_statement(speedup_df):
    if speedup_df.empty:
        return "Not enough benchmark data to decide."
    best = speedup_df.sort_values("total_speedup_x", ascending=False).iloc[0]
    mapping = {
        "priority_ranking": "risk / priority scoring",
        "rolling_alert": "time-series alerting",
        "dashboard_summary": "dashboard backend",
        "risk_model": "risk scoring / classification",
    }
    story = mapping.get(best["task"], best["task"])
    return f"""
Best end-to-end GPU story: {story} ({best['task']})

Why:
- highest total speedup in this test harness
- task is decision-facing, not just model-facing
- easy to explain in a 3-minute demo
- compatible with pandas -> cuDF / cuML acceleration

Build the pitch around:
- a LLM-powered decision tool
- GPU-native data processing
- ranked output, alert table, or forecast

Avoid:
- clustering as the core story
- token generation speed alone
- generic 'make the model faster' claims
""".strip()

decision_text = decide_problem_statement(speedup_df)
print(decision_text)


In [ ]:
bench_df.to_csv("datasense_final_benchmark_results.csv", index=False)
if not speedup_df.empty:
    speedup_df.to_csv("datasense_final_speedups.csv", index=False)

with open("datasense_final_notes.txt", "w") as f:
    f.write(decide_problem_statement(speedup_df))

print("Saved:")
print("- datasense_final_benchmark_results.csv")
print("- datasense_final_speedups.csv")
print("- datasense_final_notes.txt")
